# 06 — Checkpoint save, load, and resume

- **Mapped issue:** [#13](https://github.com/majorgilles/transformer-2017-reproduction/issues/13)
- **Depends on:** `05_training_validation_cli.ipynb` / issue #12.


In [1]:
#| default_exp checkpointing


## Goal

Understand the minimal changing state needed to pause a CPU training fixture, restore that state into newly constructed objects, and continue with the next optimizer step.


## Save the state needed to resume

A resumable checkpoint stores model parameters, optimizer progress, the
completed training step, and the PyTorch random-number-generator state in one
plain dictionary.

In [2]:
#| export
from pathlib import Path

import torch

from transformer_2017_reproduction.model import Transformer


def save_checkpoint(
    path: Path,
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    step: int,
) -> None:
    """Save the minimal CPU training state needed to resume."""
    if step < 0:
        raise ValueError("step must be non-negative")

    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step": step,
        "torch_rng_state": torch.get_rng_state(),
    }

    torch.save(checkpoint, path)

## Restore state into new objects

Loading does not recreate the model architecture. The caller first constructs
a compatible model and optimizer, then the checkpoint copies saved state into
those objects and returns the completed training step.

In [3]:
#| export
def load_checkpoint(
    path: Path,
    model: Transformer,
    optimizer: torch.optim.Optimizer,
) -> int:
    """Restore minimal CPU training state and return the saved step."""
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True,
    )

    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    torch.set_rng_state(checkpoint["torch_rng_state"])

    return checkpoint["step"]

## Save, reload, and continue training

This round-trip test trains through step 1, saves the changing training state, restores it into new model and optimizer objects, and performs step 2. It checks the saved parameters, Adam step, training step, and PyTorch random state without adding a second uninterrupted branch.


In [4]:
fixture_batches = [
    (
        torch.tensor([[2, 5, 6, 3]], dtype=torch.long),
        torch.tensor([[2, 7, 8, 3]], dtype=torch.long),
    )
]

In [5]:
torch.manual_seed(0)

uninterrupted_model = Transformer(
    vocab_size=32,
    d_model=8,
    num_heads=2,
    d_ff=16,
    max_length=8,
    pad_token_id=0,
    dropout=0.1,
    num_layers=6,
)

uninterrupted_optimizer = torch.optim.Adam(
    uninterrupted_model.parameters(),
    lr=1e-3,
    betas=(0.9, 0.98),
    eps=1e-9,
)

In [6]:
from tempfile import TemporaryDirectory

from transformer_2017_reproduction.training import train

with TemporaryDirectory() as temporary_directory:
    checkpoint_path = Path(temporary_directory) / "checkpoint.pt"

    # Train and save after step 1.
    train(
        uninterrupted_model,
        fixture_batches,
        uninterrupted_optimizer,
    )

    saved_parameters = [
        parameter.detach().clone()
        for parameter in uninterrupted_model.parameters()
    ]

    save_checkpoint(
        checkpoint_path,
        uninterrupted_model,
        uninterrupted_optimizer,
        step=1,
    )

    # These are the random values expected immediately after the save.
    expected_random_values = torch.rand(3)

    # New objects stand in for objects constructed after a restart.
    resumed_model = Transformer(
        vocab_size=32,
        d_model=8,
        num_heads=2,
        d_ff=16,
        max_length=8,
        pad_token_id=0,
        dropout=0.1,
        num_layers=6,
    )

    resumed_optimizer = torch.optim.Adam(
        resumed_model.parameters(),
        lr=1e-3,
        betas=(0.9, 0.98),
        eps=1e-9,
    )

    loaded_step = load_checkpoint(
        checkpoint_path,
        resumed_model,
        resumed_optimizer,
    )

    loaded_parameters_match = all(
        torch.equal(saved, loaded)
        for saved, loaded in zip(
            saved_parameters,
            resumed_model.parameters(),
            strict=True,
        )
    )
    loaded_optimizer_steps = {
        int(state["step"].item())
        for state in resumed_optimizer.state.values()
    }
    restored_random_values = torch.rand(3)

    assert loaded_step == 1
    assert loaded_parameters_match
    assert loaded_optimizer_steps == {1}
    assert torch.equal(restored_random_values, expected_random_values)

    # Continue for one additional step.
    resumed_loss = train(
        resumed_model,
        fixture_batches,
        resumed_optimizer,
    )
    continued_optimizer_steps = {
        int(state["step"].item())
        for state in resumed_optimizer.state.values()
    }

    assert torch.isfinite(torch.tensor(resumed_loss))
    assert continued_optimizer_steps == {2}

    print(f"Loaded training step: {loaded_step}")
    print(f"Loaded parameters match: {loaded_parameters_match}")
    print(f"Loaded Adam steps: {loaded_optimizer_steps}")
    print(
        "Random state restored: "
        f"{torch.equal(restored_random_values, expected_random_values)}"
    )
    print(f"Adam steps after continuing: {continued_optimizer_steps}")
    print(f"Resumed loss: {resumed_loss}")

assert not checkpoint_path.exists()


Loaded training step: 1
Loaded parameters match: True
Loaded Adam steps: {1}
Random state restored: True
Adam steps after continuing: {2}
Resumed loss: 5.132927417755127


## Explicitly deferred

Exact uninterrupted-versus-resumed equivalence, fresh-process orchestration, mixed-precision scaler state, CUDA random state, atomic replacement, retention policies, versioned schemas, migration support, and corruption testing are outside this minimal CPU fixture.


## Completion evidence

The fixture saves model parameters, Adam state, the completed training step, and the CPU random state. Loading restores those values into newly constructed objects, and one additional training pass advances Adam from step 1 to step 2.

The comparison demonstrates a save/load round trip in one notebook process. It does not claim bit-for-bit equivalence with an uninterrupted run or prove a fresh-process restart.
